# 03 · Exploring GSE298774 (nevus → melanoma transition, 10x Visium CytAssist)

Same task and structure as `01_explore_GSE316760.ipynb`, but this dataset is much bigger and organised differently,
so sections 1–3 first explain **how the files map to patients**.

**The study** ([PMID 42258703](https://pubmed.ncbi.nlm.nih.gov/42258703/)): about one third of melanomas arise
from a benign mole (nevus). The authors took lesions where **nevus and melanoma sit next to each other in the same
tissue section** and profiled them with Visium (plus imaging mass cytometry, not on GEO). Each lesion therefore has
a benign part and a malignant part, a natural *tumour vs non-tumour* comparison within one tissue.

### What is different from GSE316760

| | GSE316760 (notebook 01) | **GSE298774 (this notebook)** |
|---|---|---|
| Size | 2 samples | **11 GEO samples = 11 capture areas holding 19 patients' lesions (18 usable)** |
| One GEO sample = | one tissue section | **one capture area, shared by 1–3 patients** |
| Protocol / Space Ranger | Visium FFPE, 2.0.1 | **CytAssist, 2.1.1** (GEO says the data were then filtered with *semla*, but the files are not: section 7) |
| Quality | homogeneous | **very variable between patients** (median 128 to 21,899 counts per spot) |
| Capture area | 6.5 × 6.5 mm (4,992 spots) | **mostly 11 × 11 mm (14,336 spots)**, some 6.5 mm |
| Gene panel | 17,943 genes | **37,082 genes (most patients) or 18,085 (A16, A21, A23)**, both with `MT-` genes |
| Matrix | only spots under tissue | **often the whole capture area**; each patient's `tissue_positions.csv` marks its own spots (`in_tissue == 1`) |
| Problems | none | **patient A2 has no barcodes file → cannot be placed in space, excluded** |
| Units | 65 µm rule and grid agree | **65 µm rule ~7–8% off (in opposite directions for 6.5 and 11 mm areas) → grid method only** |
| Region labels | none | none: nevus vs melanoma areas are not marked |

In [ ]:
import re
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)   # harmless deprecation notices from squidpy/anndata

# Project root = the spatial-reasoning folder (the one with pyproject.toml), wherever the notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "01_data_exploration"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.io import load_visium, read_series_matrix
from space_mel.spatial import units_table

GSE = "GSE298774"
RAW = ROOT / f"data/raw/{GSE}/files"
PROCESSED = ROOT / "data/processed"
FIGS = STAGE / f"results/{GSE}"
FIGS.mkdir(parents=True, exist_ok=True)
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

## 1. How the files map to patients

GEO describes each sample as e.g. *"Nevus and melanoma in a single slide of patient A22 and A24"*: several patients'
sections were placed on **one capture area** and sequenced as one library. The authors then split the data per
patient, so the file names are `GSM<sample>_<patient>_<file>`. We list them:

In [ ]:
meta = read_series_matrix(ROOT / f"data/raw/{GSE}/metadata/{GSE}_series_matrix.txt.gz")
patients = defaultdict(dict)
for f in sorted(RAW.glob("GSM*")):
    m = re.match(r"(GSM\d+)_(.+?)_(barcodes|features|matrix|tissue_positions)", f.name)
    if m:
        patients[(m.group(1), m.group(2))][m.group(3)] = f

rows = []
for (gsm, pid), fs in patients.items():
    pos = pd.read_csv(fs["tissue_positions"])
    rows.append({
        "GEO sample": gsm, "GEO title": meta.index[meta.geo_accession == gsm][0],
        "patient": pid, "capture area spots": len(pos), "spots of this patient (in_tissue)": int(pos.in_tissue.sum()),
        "has barcodes file": "barcodes" in fs,
    })
files_table = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 80):
    display(files_table)

**Reading this:**
- 11 GEO samples, **19 patient lesions**. 4,992 spots = 6.5 mm capture area; 14,336 spots = 11 mm capture area.
- When patients share a capture area, each one's `tissue_positions.csv` lists **all** spots of the area, and
  `in_tissue == 1` marks only that patient's spots. The masks never overlap (checked: 0 shared spots).
- For most shared areas the **count matrix is the same whole-area matrix** for every patient, so to get one patient
  we load it and keep its `in_tissue` spots (`load_visium(..., in_tissue_only=True)`).
- **A2 has no `barcodes.tsv.gz`.** Its matrix has 13,916 unnamed columns, so we cannot know which column is which
  spot. A2 is **excluded** (worth asking Myles / the authors). A2's gene list was uploaded separately
  (`GSE298774_A2_features.tsv.gz`) but that does not solve the missing barcodes.

## 2. A shared capture area, visualised

Capture area GSM9022967 holds patients **A22** and **A24**. Below: every spot of the area, coloured by whose
`in_tissue` mask it belongs to. Grey spots are on the capture area but under neither patient's tissue (they
are in the matrix, but are not part of either patient's data).

In [ ]:
def plot_shared_area(gsm, pids, ax):
    first = load_visium(RAW, f"{gsm}_{pids[0]}", PROCESSED / f"visium/{GSE}_{pids[0]}", library_id=pids[0])
    lib = first.uns["spatial"][pids[0]]
    scale = lib["scalefactors"]["tissue_hires_scalef"]
    ax.imshow(lib["images"]["hires"])
    pos0 = pd.read_csv(patients[(gsm, pids[0])]["tissue_positions"], index_col=0)
    ax.scatter(pos0.pxl_col_in_fullres * scale, pos0.pxl_row_in_fullres * scale, s=1, c="lightgrey", label="no patient")
    for pid in pids:
        pos = pd.read_csv(patients[(gsm, pid)]["tissue_positions"], index_col=0)
        pos = pos[pos.in_tissue == 1]
        ax.scatter(pos.pxl_col_in_fullres * scale, pos.pxl_row_in_fullres * scale, s=1.5, label=pid)
    ax.legend(markerscale=6, loc="lower right")
    ax.set_title(f"{gsm}: {', '.join(pids)}")
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_shared_area("GSM9022967", ["A22", "A24"], axes[0])
plot_shared_area("GSM9022972", ["B14", "B15", "B38_2"], axes[1])
fig.savefig(FIGS / "shared_capture_areas.png", bbox_inches="tight")

Each patient's section is a separate piece of tissue on the same capture area. Inside each piece there should be a
nevus part and a melanoma part, but **GEO does not say which spots are which**.

## 3. Load all patients

One AnnData per patient, keeping only that patient's spots. Patients with whole-area matrices take a few seconds
each because the full matrix is read before filtering.

In [ ]:
adatas = {}
for (gsm, pid), fs in patients.items():
    if "barcodes" not in fs:
        print(f"skipping {pid}: no barcodes file")
        continue
    adatas[pid] = load_visium(RAW, f"{gsm}_{pid}", PROCESSED / f"visium/{GSE}_{pid}", library_id=pid,
                              in_tissue_only=True)
    adatas[pid].uns["geo_sample"] = gsm
print(len(adatas), "patients loaded")

## 4. What is inside, and metadata

In [ ]:
overview = pd.DataFrame({
    pid: {
        "GEO sample": a.uns["geo_sample"],
        "spots": a.n_obs,
        "genes in panel": a.n_vars,
        "MT- genes": int(a.var_names.str.startswith("MT-").sum()),
        "capture area": "11 mm" if len(pd.read_csv(patients[(a.uns["geo_sample"], pid)]["tissue_positions"])) > 5000 else "6.5 mm",
        "integer counts": bool(np.allclose(a.X.data[:1000], np.round(a.X.data[:1000]))),
    }
    for pid, a in adatas.items()
}).T
overview

In [ ]:
with pd.option_context("display.max_colwidth", 400):
    display(meta[["data_processing"]].head(1))

**Reading this:**
- 18 usable patients, from ~1,100 to ~6,400 spots each.
- **Two gene panels.** Most patients have 37,082 genes. A16, A21 and A23 have 18,085 (the probe-set v2 panel, as in
  GSE300445). To compare patients we will have to keep the genes they share.
- Space Ranger 2.1.1 with CytAssist, then filtered with *semla*: "spots with more than 500 genes … and less than
  30% mitochondrial counts" were kept. We check that in section 7.

## 5. Tissue images (one per capture area)

In [ ]:
first_per_area = {}
for pid, a in adatas.items():
    first_per_area.setdefault(a.uns["geo_sample"], pid)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax in axes.flat:
    ax.axis("off")
for ax, (gsm, pid) in zip(axes.flat, first_per_area.items()):
    ax.imshow(adatas[pid].uns["spatial"][pid]["images"]["hires"])
    others = [p for (g, p) in patients if g == gsm]
    ax.set_title(f"{gsm}: {', '.join(others)}")
fig.savefig(FIGS / "tissue_images.png", bbox_inches="tight")

## 6. Coordinate units

Same check as notebooks 01 and 02, with the shared helper. Micrometre coordinates go to `obsm["spatial_um"]`.

In [ ]:
units = units_table(adatas)
units

**Conclusion:** the 65 µm rule is off by ~7% **in opposite directions**: −7.1% on the 6.5 mm capture areas (A7, A21,
D3, D20; as in GSE300445) and +7.6 to +7.9% on the 11 mm ones. The grid method is consistent everywhere. Pixel sizes
vary a lot between capture areas, so **micrometres are essential** for comparing distances across patients.
Patients sharing a capture area share the same pixel size (same image).

## 7. Quality control

In [ ]:
for pid, a in adatas.items():
    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], inplace=True, percent_top=None)

qc = pd.concat({pid: a.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]] for pid, a in adatas.items()},
               names=["patient"]).reset_index(level=0)
summary = qc.groupby("patient", sort=False).agg(
    median_counts=("total_counts", "median"), median_genes=("n_genes_by_counts", "median"),
    min_genes=("n_genes_by_counts", "min"), max_pct_mt=("pct_counts_mt", "max"))
display(summary.round(1))

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
for ax, col in zip(axes, ["total_counts", "n_genes_by_counts", "pct_counts_mt"]):
    # spots with 0 counts have an undefined % MT (NaN); leave them out of the boxes
    ax.boxplot([qc.loc[qc.patient == p, col].dropna() for p in adatas], showfliers=False)
    ax.set_ylabel(col)
axes[0].set_yscale("log")   # counts span 3 orders of magnitude between patients
axes[-1].set_xticks(range(1, len(adatas) + 1), list(adatas), rotation=45)
fig.savefig(FIGS / "qc_by_patient.png", bbox_inches="tight")

**Was the authors' filter applied?** GEO says spots with ≤ 500 genes or ≥ 30% mitochondrial counts were removed with
*semla*. If that were true of these files, `min_genes` would be > 500 and `max_pct_mt` < 30 everywhere. Counting the
spots that would fail that filter:

In [ ]:
fails = qc.assign(
    zero_counts=qc.total_counts == 0,
    under_500_genes=qc.n_genes_by_counts <= 500,
    mt_30_or_more=qc.pct_counts_mt >= 30,
).groupby("patient", sort=False)[["zero_counts", "under_500_genes", "mt_30_or_more"]].mean().mul(100).round(1)
fails.columns = ["% spots with 0 counts", "% spots with <= 500 genes", "% spots with >= 30% MT"]
fails

**Reading this:** the GEO files are only **partly** filtered as described. The mitochondrial threshold holds
(practically no spot has ≥ 30% MT), and spots with no counts at all are extremely rare. But the **500-gene threshold
was clearly not applied**: in many patients most spots have ≤ 500 genes (85% in B8, 58–70% in A17, A34, B13, B14 and
B15). Data quality differs enormously between patients (median counts per spot from ~130 to ~22,000, see the table
and the log-scale boxplot), so shallow patients will give noisy marker maps and clusters. Before spatial analyses we
will need **our own QC filter** (and possibly exclude the weakest patients).

## 8. Marker genes

Same markers as notebook 02. **`PRAME` is especially useful here:** pathologists use it to tell melanoma (PRAME
high) from nevus (PRAME low), and both melanocytic markers (`MLANA`, `PMEL`) are high in both. So PRAME is our
first, rough way to find the melanoma part of each lesion.

First `PRAME` and `MLANA` across all patients, then the full panel for four example patients.

In [ ]:
MARKERS = ["MLANA", "PMEL", "SOX10", "PRAME", "MKI67", "PTPRC", "CD3E", "CD8A", "CD68", "CD163", "CD74",
           "COL1A1", "KRT14"]

for a in adatas.values():
    a.layers["counts"] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

for gene in ["MLANA", "PRAME"]:
    # one shared colour scale (0 to the 99th percentile over all patients) so patients can be compared
    vmax = np.percentile(np.concatenate([a[:, gene].X.toarray().ravel() for a in adatas.values()]), 99)
    fig, axes = plt.subplots(4, 5, figsize=(22, 18))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (pid, a) in zip(axes.flat, adatas.items()):
        sq.pl.spatial_scatter(a, library_id=pid, color=gene, img=False, ax=ax, title=f"{pid}: {gene}",
                              colorbar=False, vmin=0, vmax=vmax)
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, vmax), cmap="viridis"), ax=axes, shrink=0.4,
                 label=f"{gene}, log-normalised expression (same scale for all patients)")
    fig.savefig(FIGS / f"{gene}_all_patients.png", bbox_inches="tight")
    plt.show()

In [ ]:
EXAMPLES = ["A22", "A24", "B4", "A16"]   # A22/A24 share a capture area; B4 is the largest; A16 has the 18k panel
for pid in EXAMPLES:
    sq.pl.spatial_scatter(adatas[pid], library_id=pid, color=MARKERS, img=False, ncols=7, figsize=(3, 3),
                          save=FIGS / f"markers_{pid}.png")

## 9. A first data-driven segmentation (clustering), example patients

Same procedure as notebooks 01/02 for the four example patients.

In [ ]:
for pid in EXAMPLES:
    a = adatas[pid]
    sc.pp.highly_variable_genes(a, n_top_genes=2000, flavor="seurat_v3", layer="counts")
    sc.pp.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a)
    sc.tl.leiden(a, resolution=0.5, flavor="igraph", n_iterations=2, key_added="leiden")
    sc.tl.umap(a)
    print(pid, a.obs.leiden.value_counts().sort_index().to_dict())

In [ ]:
for pid in EXAMPLES:
    a = adatas[pid]
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    sc.pl.umap(a, color="leiden", ax=axes[0], show=False, title=f"{pid}: UMAP")
    sq.pl.spatial_scatter(a, library_id=pid, color="leiden", ax=axes[1], title=f"{pid}: clusters in space")
    fig.savefig(FIGS / f"clusters_{pid}.png", bbox_inches="tight")

In [ ]:
for pid in EXAMPLES:
    a = adatas[pid]
    sc.pl.matrixplot(a, MARKERS, groupby="leiden", standard_scale="var", title=pid, cmap="viridis", show=False)
    plt.savefig(FIGS / f"cluster_markers_{pid}.png", bbox_inches="tight")
    plt.show()
    sc.tl.rank_genes_groups(a, "leiden", method="wilcoxon")
    print(f"--- {pid}: top 8 genes per cluster")
    display(pd.DataFrame(a.uns["rank_genes_groups"]["names"]).head(8))

## 10. Save

One compressed `.h5ad` per patient in `data/processed/` (gitignored): raw counts in `layers["counts"]`,
normalised values in `X`, QC metrics, µm coordinates, and clusters for the example patients.

In [ ]:
for pid, a in adatas.items():
    path = PROCESSED / f"{GSE}_{pid}.h5ad"
    a.write_h5ad(path, compression="gzip")
print(len(adatas), "files saved, e.g.", path.relative_to(ROOT), f"({path.stat().st_size / 1e6:.0f} MB)")

## 11. Summary and open questions

**What the data contains (task checklist)**
- **Expression matrices:** 11 GEO samples = 11 capture areas holding **19 patients' lesions; 18 usable** (A2 has no
  barcodes). Patients sharing an area come with the whole-area matrix and are split by their `in_tissue` mask.
  Two panels: 37,082 genes (15 patients) and 18,085 (A16, A21, A23).
- **Metadata:** CytAssist, Space Ranger 2.1.1. GEO says spots were filtered with *semla* (> 500 genes, < 30% MT):
  the MT threshold holds, but the gene threshold was **not** applied (up to 85% of spots below it in B8).
- **Spatial coordinates:** **0.22–0.39 µm/px** from the grid. The 65 µm rule is −7.1% on 6.5 mm areas and
  +7.6 to +7.9% on 11 mm areas. µm in `obsm["spatial_um"]`.
- **Tissue images:** one H&E per capture area, shared by its patients; larger than the capture area.
- **Annotations:** none: nevus and melanoma parts are not marked.
- **Quality:** ~170-fold difference in depth between patients (median 128 counts/spot in B8 to 21,899 in D20).
  Very shallow: B8, B14, B15, A17, A7, A34 (and B13).

**First biological look** (hypotheses to check)
- **PRAME** (common colour scale) forms coherent positive regions in A24 and A16, and moderate ones in A21, A22, B1,
  B4, D20, D3 and at the rim of A17: candidate **melanoma parts**; the PRAME-negative melanocytic areas are candidate
  **nevus parts**. PRAME is nearly absent in A7, A34, B8, B13, B14, B15, which are mostly the shallow patients, so
  the absence may be technical. Isolated bright spots in shallow patients are noise.
- **A24** is the best tumour–immune example: a PRAME+ tumour region, an **IFN-γ / T-cell inflamed cluster** (CXCL9,
  CXCL10, STAT1, GBP1/5) and many plasma cells.
- Several patients have 2–3 melanocytic clusters (A16, A22, A24, B4): candidates for nevus vs melanoma, or for
  different melanoma cell states (e.g. A16 cluster 5, neural-crest-like: RXRG, PTPRZ1, PLP1).

**Open questions / next steps**
1. Obtain the authors' nevus / melanoma annotations (paper supplement or contact).
2. Ask whether A2's barcodes can be recovered.
3. Decide our own QC thresholds and which patients to keep (exclude the shallowest?).
4. Combine PRAME and MLANA scores to delineate melanoma vs nevus regions per lesion.